In [ ]:
import pandas as pd
import numpy as np

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", None)

In [19]:
nba = pd.read_csv("NBA_Master_Dataset_2026.csv")

nba.head()

,Team,AvgAge,Payroll,DeadCash,CapAllocations,Cap Space,DeadCap,PTS,FG%,3P%,FT%,TRB,AST,STL,BLK,TOV,RosterAge,W,L,MOV,SRS,ORtg,DRtg,NRtg,Pace,TS%,WinPct,CostPerWin,PayrollRank,CapAllocationRank,PointDiffPerDollar,NetRatingPerMillion
0,OKC,24.6,187827634,2298085,188057857,-33410857,2296274,119.0,0.484,0.365,0.817,44.1,25.8,9.7,5.5,12.6,25.2,64.0,18.0,11.15,11.04,118.9,107.7,11.2,99.3,0.599,0.780,2934806.781,19.0,23.0,0.059,0.060
1,SAS,26.6,187078708,7569181,183990669,-29343669,7258201,119.8,0.483,0.359,0.787,47.0,28.1,7.5,5.5,13.5,25.4,62.0,20.0,8.30,8.28,119.6,111.3,8.3,99.9,0.595,0.756,3017398.516,21.0,25.0,0.044,0.044
2,DET,25.9,179870831,6210275,193082507,-38435507,9337154,117.8,0.485,0.356,0.763,45.6,27.8,10.4,6.4,15.1,26.1,60.0,22.0,8.16,7.53,117.9,109.7,8.2,99.3,0.583,0.732,2997847.183,25.0,21.0,0.045,0.046
3,BOS,26.1,196926936,582842,194526296,-39879296,469063,114.9,0.467,0.367,0.807,46.4,24.6,7.1,5.0,12.4,26.9,56.0,26.0,7.70,7.37,120.8,112.7,8.1,94.8,0.583,0.683,3516552.429,10.0,18.0,0.039,0.041
4,DEN,26.7,192006289,636435,200743895,-46096895,0,122.1,0.496,0.396,0.808,44.0,29.0,6.8,4.0,12.9,27.8,54.0,28.0,5.15,4.97,122.6,117.4,5.2,98.4,0.616,0.659,3555672.019,12.0,11.0,0.027,0.027


In [20]:
print(nba.columns.tolist())

['Team', 'AvgAge', 'Payroll', 'DeadCash', 'CapAllocations', 'Cap Space', 'DeadCap', 'PTS', 'FG%', '3P%', 'FT%', 'TRB', 'AST', 'STL', 'BLK', 'TOV', 'RosterAge', 'W', 'L', 'MOV', 'SRS', 'ORtg', 'DRtg', 'NRtg', 'Pace', 'TS%', 'WinPct', 'CostPerWin', 'PayrollRank', 'CapAllocationRank', 'PointDiffPerDollar', 'NetRatingPerMillion']


In [21]:
# Money spent for every win
nba["MoneyPerWin"] = nba["Payroll"] / nba["W"]

# Wins produced per $1 million
nba["WinsPerMillion"] = nba["W"] / (nba["Payroll"] / 1_000_000)

# Dead money as % of payroll
nba["DeadMoneyPercent"] = (
    nba["DeadCash"] / nba["Payroll"]
) * 100

# Cap allocation as % of payroll
nba["CapAllocationPercent"] = (
    nba["CapAllocations"] / nba["Payroll"]
) * 100

# Available cap space as % of payroll
nba["CapSpacePercent"] = (
    nba["Cap Space"] / nba["Payroll"]
) * 100

# Offensive Rating per million dollars
nba["ORtgPerMillion"] = (
    nba["ORtg"] /
    (nba["Payroll"] / 1_000_000)
)

# Defensive Rating per million dollars
# Lower DRtg is better, so invert it
nba["DRtgPerMillion"] = (
    (120 - nba["DRtg"]) /
    (nba["Payroll"] / 1_000_000)
)

# Net Rating per million
nba["NetRatingPerMillion2"] = (
    nba["NRtg"] /
    (nba["Payroll"] / 1_000_000)
)

# Points scored per million dollars
nba["PointsPerMillion"] = (
    nba["PTS"] /
    (nba["Payroll"] / 1_000_000)
)

# True Shooting per million
nba["TSPerMillion"] = (
    nba["TS%"] /
    (nba["Payroll"] / 1_000_000)
)

# Wins per average age
nba["AgeEfficiency"] = (
    nba["W"] /
    nba["AvgAge"]
)

# Cap flexibility score
nba["CapFlexibilityScore"] = (
    nba["Cap Space"] -
    nba["DeadCash"]
)

# Payroll efficiency score
nba["PayrollEfficiency"] = (
    nba["WinPct"] /
    (nba["Payroll"] / 1_000_000)
)

# Net Rating per payroll dollar
nba["NetRatingPerDollar"] = (
    nba["NRtg"] /
    nba["Payroll"]
)

In [22]:
nba["SalaryEfficiencyIndex"] = (

      0.30 * nba["WinsPerMillion"]

    + 0.25 * nba["PayrollEfficiency"]

    + 0.20 * nba["NetRatingPerMillion"]

    + 0.15 * nba["PointDiffPerDollar"]

    - 0.10 * nba["DeadMoneyPercent"]

)

In [23]:
def ranking_table(df, column, ascending):

    best = (
        df.sort_values(column, ascending=ascending)
        [["Team", column]]
        .head(5)
        .reset_index(drop=True)
    )

    worst = (
        df.sort_values(column, ascending=ascending)
        [["Team", column]]
        .tail(5)
        .iloc[::-1]
        .reset_index(drop=True)
    )

    table = pd.DataFrame({

        "Best Team": best["Team"],

        "Best Value": best[column],

        "Worst Team": worst["Team"],

        "Worst Value": worst[column]

    })

    return table

In [24]:
categories = {

    "Dead Money": ("DeadCash", True),

    "Dead Money %": ("DeadMoneyPercent", True),

    "Money Per Win": ("MoneyPerWin", True),

    "Wins Per Million": ("WinsPerMillion", False),

    "Cap Allocations": ("CapAllocations", True),

    "Cap Allocation %": ("CapAllocationPercent", True),

    "Cap Space": ("Cap Space", False),

    "Cap Space %": ("CapSpacePercent", False),

    "Payroll Efficiency": ("PayrollEfficiency", False),

    "Cost Per Win": ("CostPerWin", True),

    "Point Differential Per Dollar": ("PointDiffPerDollar", False),

    "Net Rating Per Million": ("NetRatingPerMillion", False),

    "Offensive Rating Per Million": ("ORtgPerMillion", False),

    "Defensive Rating Per Million": ("DRtgPerMillion", False),

    "Points Per Million": ("PointsPerMillion", False),

    "True Shooting Per Million": ("TSPerMillion", False),

    "Age Efficiency": ("AgeEfficiency", False),

    "Cap Flexibility": ("CapFlexibilityScore", False),

    "Overall Salary Efficiency": ("SalaryEfficiencyIndex", False)

}

In [25]:
dashboard.style \
.hide(axis="index") \
.set_caption("🏀 NBA Salary Efficiency Dashboard") \
.set_properties(**{
    "text-align":"center",
    "font-size":"12pt",
    "border":"1px solid black"
}) \
.set_table_styles([
    {
        "selector":"th",
        "props":[
            ("background-color","#1D428A"),
            ("color","white"),
            ("font-size","14pt"),
            ("text-align","center")
        ]
    },
    {
        "selector":"tbody tr:nth-child(even)",
        "props":[
            ("background-color","#F7F7F7")
        ]
    }
]) \
.apply(color_best_worst)

AttributeError: The '.style' accessor requires jinja2

In [9]:
dashboard = pd.DataFrame({"Rank": [1,2,3,4,5]})

for title, (column, ascending) in categories.items():

    rankings = ranking_table(nba, column, ascending)

    dashboard[f"{title} (Best)"] = rankings["Best Team"]

    dashboard[f"{title} (Worst)"] = rankings["Worst Team"]

dashboard

,Rank,Dead Money (Best),Dead Money (Worst),Dead Money % (Best),Dead Money % (Worst),Money Per Win (Best),Money Per Win (Worst),Wins Per Million (Best),Wins Per Million (Worst),Cap Allocations (Best),Cap Allocations (Worst),Cap Allocation % (Best),Cap Allocation % (Worst),Cap Space (Best),Cap Space (Worst),Cap Space % (Best),Cap Space % (Worst),Payroll Efficiency (Best),Payroll Efficiency (Worst),Cost Per Win (Best),Cost Per Win (Worst),Point Differential Per Dollar (Best),Point Differential Per Dollar (Worst),Net Rating Per Million (Best),Net Rating Per Million (Worst),Offensive Rating Per Million (Best),Offensive Rating Per Million (Worst),Defensive Rating Per Million (Best),Defensive Rating Per Million (Worst),Points Per Million (Best),Points Per Million (Worst),True Shooting Per Million (Best),True Shooting Per Million (Worst),Age Efficiency (Best),Age Efficiency (Worst),Cap Flexibility (Best),Cap Flexibility (Worst),Overall Salary Efficiency (Best),Overall Salary Efficiency (Worst)
0,1,HOU,MIL,HOU,BKN,OKC,WAS,OKC,WAS,BKN,GSW,MEM,WAS,BKN,GSW,BKN,WAS,OKC,WAS,OKC,WAS,OKC,WAS,OKC,BKN,UTA,DAL,OKC,UTA,UTA,DAL,UTA,DAL,OKC,WAS,MEM,WAS,HOU,BKN
1,2,LAC,MIA,NYK,MIL,DET,IND,DET,IND,MEM,WAS,DAL,UTA,MEM,WAS,MEM,GSW,DET,IND,DET,IND,DET,BKN,DET,WAS,BKN,CLE,DET,WAS,BKN,CLE,BKN,CLE,SAS,IND,BKN,GSW,BOS,MIL
2,3,NYK,POR,LAC,MIA,SAS,SAC,SAS,SAC,UTA,MIN,SAS,SAC,UTA,MIN,UTA,MIN,SAS,SAC,SAS,SAC,SAS,UTA,SAS,UTA,CHA,GSW,SAS,SAC,DET,GSW,CHA,GSW,DET,SAC,OKC,PHX,NYK,MIA
3,4,SAC,PHX,GSW,POR,BOS,DAL,BOS,DAL,MIL,CLE,BOS,MIN,MIL,CLE,MIL,CLE,BOS,DAL,BOS,DAL,BOS,SAC,BOS,SAC,DET,NYK,BOS,MIL,CHA,NYK,WAS,NYK,BOS,BKN,UTA,MIN,DEN,POR
4,5,GSW,BKN,SAC,WAS,DEN,BKN,DEN,BKN,CHA,SAC,NYK,GSW,CHA,SAC,SAS,SAC,DEN,BKN,DEN,BKN,NYK,IND,NYK,IND,WAS,PHI,TOR,NOP,WAS,LAC,MIL,PHI,NYK,UTA,SAS,CLE,LAC,WAS


In [10]:
for title, (column, ascending) in categories.items():

    print("=" * 90)
    print(title.upper())
    print("=" * 90)

    display(ranking_table(nba, column, ascending))

DEAD MONEY


,Best Team,Best Value,Worst Team,Worst Value
0,HOU,110300,MIL,28508207
1,LAC,363480,MIA,26643031
2,NYK,369424,POR,26455281
3,SAC,495591,PHX,24209031
4,GSW,529164,BKN,23426027


DEAD MONEY %


,Best Team,Best Value,Worst Team,Worst Value
0,HOU,0.055621,BKN,16.232762
1,NYK,0.175025,MIL,15.710653
2,LAC,0.182988,MIA,14.152218
3,GSW,0.251959,POR,14.022389
4,SAC,0.261468,WAS,13.186182


MONEY PER WIN


...

,Best Team,Best Value,Worst Team,Worst Value
0,OKC,2.601626,WAS,0.714286
1,SAS,2.330827,IND,0.730769
2,DET,2.316602,SAC,0.805861
3,BOS,2.145594,BKN,0.854701
4,NYK,2.030651,UTA,0.876494


CAP FLEXIBILITY


,Best Team,Best Value,Worst Team,Worst Value
0,MEM,-15945939,WAS,-100251473
1,BKN,-19739379,GSW,-80104889
2,OKC,-35708942,PHX,-77359020
3,UTA,-35870473,MIN,-75002643
4,SAS,-36912850,CLE,-73009394


OVERALL SALARY EFFICIENCY


,Best Team,Best Value,Worst Team,Worst Value
0,HOU,0.083203,BKN,-1.605827
1,BOS,0.070631,MIL,-1.529723
2,NYK,0.069293,MIA,-1.341804
3,DEN,0.061534,POR,-1.335475
4,LAC,0.047878,WAS,-1.312658
